# Notebook 2 — Can we trust the data? (2023 contract)
~40 s to run.

*What you'll see:* what the raw 2023 rows really look like, the **validation contract**
(16 rules with named owners), two findings that changed the rules, and the **PUBLISH / HOLD
gate**. Mindset: contract *before* cleaning; broken rows are quarantined with reason codes —
never deleted. A rule that *cannot be evaluated* (NULL) counts as failed, never as a pass.

**Load the validation report** the pipeline produced over all 38,310,122 rows.

In [1]:
import json, sys
from pathlib import Path
import pandas as pd
import duckdb
pd.set_option("display.width", 175)

for base in [Path.cwd(), Path.cwd().parent]:
    if (base / "src" / "nyc_pipeline").exists():
        sys.path.insert(0, str(base / "src")); ROOT = base; break

report = json.loads((ROOT / "output/validation_report.json").read_text())
t = report["totals"]
print(f"raw {t['raw_rows']:,} → valid {t['valid_rows']:,} ({t['coverage']:.2%}) | "
      f"quarantined {t['quarantined_rows']:,} (kept, with reason codes)")

raw 38,310,122 → valid 37,532,268 (97.97%) | quarantined 777,854 (kept, with reason codes)


**Profile the raw month** — what the source actually contains, before we judge it.
One row per trip, no coordinates, TLC's own zone ids.

In [2]:
cols = pd.DataFrame([{
    "column": c, "type": str(d),
} for c, d in duckdb.sql(
    f"DESCRIBE SELECT * FROM read_parquet('{ROOT}/data/raw/yellow_tripdata_2023-01.parquet')"
).df()[["column_name", "column_type"]].values])
display(cols)
print("19 columns · no longitude/latitude · zone ids are TLC's own (pulocationid/dolocationid)")

,column,type
0,vendorid,BIGINT
1,tpep_pickup_datetime,TIMESTAMP
2,tpep_dropoff_datetime,TIMESTAMP
3,passenger_count,BIGINT
4,trip_distance,DOUBLE
5,ratecodeid,BIGINT
6,store_and_fwd_flag,VARCHAR
7,pulocationid,BIGINT
8,dolocationid,BIGINT
9,payment_type,BIGINT


19 columns · no longitude/latitude · zone ids are TLC's own (pulocationid/dolocationid)


**Domains of the categorical fields** for one month — the evidence behind the rule
tolerances. Every rate below is a decision input, not a guess.

In [3]:
q = lambda s: duckdb.sql(s).df()
dom = q(f"""
SELECT ratecodeid, vendorid, payment_type, count(*) AS trips
FROM read_parquet('{ROOT}/data/raw/yellow_tripdata_2023-01.parquet')
GROUP BY ALL ORDER BY trips DESC LIMIT 12""")
display(dom)
print("→ ratecodeid 99 and vendor 6 are undocumented (U1/U4); payment_type 0 + missing rate code")
print("  is a 3.4% cohort that behaves like real trips (U5) — it stays in the population.")

,ratecodeid,vendorid,payment_type,trips
0,1,2,1,1673295
1,1,1,1,612730
2,1,2,2,371703
3,1,1,2,134785
4,2,2,1,70566
5,<NA>,2,0,47086
6,1,2,4,27077
7,<NA>,1,0,24657
8,2,1,1,20098
9,2,2,2,16498


→ ratecodeid 99 and vendor 6 are undocumented (U1/U4); payment_type 0 + missing rate code
  is a 3.4% cohort that behaves like real trips (U5) — it stays in the population.


**Finding 1 — the missing-metadata cohort (U5).** 1,309,356 rows (3.4%) have no rate
code, no passenger count, no store-and-fwd flag, `payment_type = 0` — and their only other
oddity is NULL in the two *newest* columns. Their trips look real, so we keep them and flag
them instead of quarantining.

In [4]:
u5 = q(f"""
SELECT count(*) AS rows,
       round(avg(trip_distance), 2) AS avg_distance_mi,
       round(quantile_cont(epoch(tpep_dropoff_datetime - tpep_pickup_datetime)/60, 0.5), 1) AS p50_min,
       round(avg(fare_amount), 2) AS avg_fare,
       sum(CASE WHEN airport_fee IS NULL THEN 1 ELSE 0 END) AS null_airport_fee,
       sum(CASE WHEN congestion_surcharge IS NULL THEN 1 ELSE 0 END) AS null_congestion_fee
FROM read_parquet('{ROOT}/data/raw/*.parquet') WHERE ratecodeid IS NULL""")
display(u5)
print("Decision (A6/A7): treat as real trips with missing metadata → keep, flag (R10/R11/R13),")
print("read a missing surcharge as 0 for the money check. Client sign-off still open (U5).")

,rows,avg_distance_mi,p50_min,avg_fare,null_airport_fee,null_congestion_fee
0,1309356,18.47,16.6,22.37,1309356.0,1309356.0


Decision (A6/A7): treat as real trips with missing metadata → keep, flag (R10/R11/R13),
read a missing surcharge as 0 for the money check. Client sign-off still open (U5).


**Finding 2 — the money check had to be rebuilt.** 2023 adds `airport_fee` and
`congestion_surcharge`, and inclusion conventions vary per row: 26.7% of rows differ from the
component sum by $1–2.50, but **none** exceed $5. So R09 uses an absolute $5 tolerance
(calibrated, not guessed) and stays a FLAG — money is not in the KPI anyway (L5).

In [5]:
comp = ("coalesce(fare_amount,0)+coalesce(extra,0)+coalesce(mta_tax,0)+coalesce(tip_amount,0)"
        "+coalesce(tolls_amount,0)+coalesce(improvement_surcharge,0)"
        "+coalesce(airport_fee,0)+coalesce(congestion_surcharge,0)")
resid = q(f"""
SELECT CASE WHEN abs(total_amount - ({comp})) <= 0.01 THEN 'exact'
            WHEN abs(total_amount - ({comp})) <= 2.5  THEN '$0.01-2.50'
            WHEN abs(total_amount - ({comp})) <= 5    THEN '$2.50-5'
            ELSE '> $5' END AS residual_band,
       count(*) AS trips
FROM read_parquet('{ROOT}/data/raw/*.parquet') GROUP BY 1 ORDER BY 2 DESC""")
display(resid)
print("→ no row is off by more than $5, so the $5 tolerance flags nothing today and still")
print("  catches a real break tomorrow. Residual detail is disclosed as limitation L5.")

,residual_band,trips
0,exact,28082903
1,$0.01-2.50,8848852
2,$2.50-5,1378367


→ no row is off by more than $5, so the $5 tolerance flags nothing today and still
  catches a real break tomorrow. Residual detail is disclosed as limitation L5.


**The contract** — 16 executable rules. QUARANTINE removes a trip from the KPI
population; FLAG keeps it and raises a hand; FAIL holds the publish gate; UNKNOWN is a question
for the client. Every rule has a named owner and its tolerance is calibrated on 2023.

In [6]:
rules = pd.DataFrame([{
    "id": r["id"], "severity": r["severity"], "status": r["status"],
    "failures": "-" if r["failures"] is None else f"{r['failures']:,}",
    "fail_rate": "-" if r["fail_rate"] is None else f"{r['fail_rate']:.3%}",
    "name": r["name"], "owner": r["owner"],
} for r in report["rules"]])
display(rules)
print("PASS = within tolerance · WARN = disclose with the KPI · FAIL = gate holds · UNKNOWN = ask")

,id,severity,status,failures,fail_rate,name,owner
0,R00,QUARANTINE,PASS,0,0.000%,timestamps_parseable,FDE (technical contract)
1,R01,QUARANTINE,PASS,"15,569",0.041%,dropoff_after_pickup,FDE (technical contract)
2,R02,QUARANTINE,PASS,212,0.001%,duration_within_24h,CLIENT — threshold is a judgement call (assump...
3,R03,QUARANTINE,PASS,0,0.000%,pickup_in_file_month,FDE (partition contract)
4,R04,QUARANTINE,PASS,0,0.000%,pickup_zone_known,"FDE (zone contract, ids from the TLC zone lookup)"
5,R05,QUARANTINE,PASS,0,0.000%,dropoff_zone_known,FDE (zone contract)
6,R06,QUARANTINE,WARN,"774,434",2.022%,distance_in_range,FDE (KPI is distance-banded: a trip without di...
7,R07,FLAG,PASS,"1,892,770",4.941%,passenger_count_plausible,CLIENT (unknown = U2)
8,R08,FLAG,PASS,"383,464",1.001%,money_non_negative,FDE / finance
9,R09,FLAG,PASS,0,0.000%,total_matches_components,finance


PASS = within tolerance · WARN = disclose with the KPI · FAIL = gate holds · UNKNOWN = ask


**Row conservation + retrieval integrity** — the two guards that catch a broken job:
nothing appeared or vanished, and every stored month still matches the API's own count.

In [7]:
cons = pd.DataFrame(report["row_conservation"])
print(f"conservation ok: {bool(cons['ok'].all())} | "
      f"staged rows == manifest rows: {bool((cons['staged_rows'] == cons['manifest_rows']).all())}")
integ = pd.DataFrame(report["integrity_checks"])
print(f"retrieval integrity ok: {bool(integ['ok'].all())} | months reconciled: "
      f"{int(integ['api_count_reconciled'].sum())}/{len(integ)} | duplicate keys: "
      f"{int(integ['composite_key_duplicates'].sum())}")

conservation ok: True | staged rows == manifest rows: True
retrieval integrity ok: True | months reconciled: 12/12 | duplicate keys: 2


**The gate** — the actual deliverable of validation: publish or hold, with reasons, plus
the questions only the client can answer and the limits no rule can fix.

In [8]:
g = report["gate"]
print(f"GATE: {g['status']}   (min coverage {g['min_valid_coverage']:.0%}, actual {t['coverage']:.2%})")
if g["reasons"]:
    print("reasons:", *g["reasons"], sep="\n  - ")
print("\nOpen questions for the client:")
for x in report["unknowns"]:
    print(f"  {x['id']}: {x['text'][:100]}…  → {x.get('owner','')}")
print("\nLimitations (no rule can fix these):")
for x in report["limitations"]:
    print(f"  {x['id']}: {' '.join(x['text'].split())[:105]}…")

GATE: PUBLISH   (min coverage 90%, actual 97.97%)

Open questions for the client:
  U1: ratecodeid = 99 is not defined in the TLC user guide (213,480 rows).…  → CLIENT / TLC research@tlc.nyc.gov
  U2: passenger_count = 0 / 7 / 8 / 9: unknown passenger, error, or courier trip? (583,414 rows)…  → CLIENT
  U3: Zero-distance trips (2.02% of rows, rising to 3.4% by Sep-Nov) — meter fault, repositioning, or canc…  → CLIENT
  U4: vendorid = 6 appears in 2023 (8,668 rows) but is not in the TLC user guide.…  → CLIENT / TLC
  U5: 3.4% of rows carry no ratecodeid / passenger_count / store_and_fwd_flag, payment_type = 0, and NULL …  → CLIENT / TLC

Limitations (no rule can fix these):
  L1: TLC did not create this data (technology providers do) and disclaims accuracy — absolute values carry unq…
  L2: This source has no coordinates: location is TLC's own zone id (1..265). 264/265 mean 'Unknown'/'NA' (abou…
  L3: No driver/vehicle IDs, no traffic/weather feeds — analysis is association-only.…
  L4: